# Question B4 (10 marks)

> Model degradation is a common issue faced when deploying machine learning models (including neural networks) in the real world. New data points could exhibit a different pattern from older data points due to factors such as changes in government policy or market sentiments. For instance, housing prices in Singapore have been increasing and the Singapore government has introduced 3 rounds of cooling measures over the past years (16 December 2021, 30 September 2022, 27 April 2023).

> In such situations, the distribution of the new data points could differ from the original data distribution which the models were trained on. Recall that machine learning models often work with the assumption that the test distribution should be similar to train distribution. When this assumption is violated, model performance will be adversely impacted.  In the last part of this assignment, we will investigate to what extent model degradation has occurred.


> Your co-investigators used a linear regression model to rapidly test out several combinations of train/test splits and shared with you their findings in a brief report attached in Appendix A below. You wish to investigate whether your deep learning model corroborates with their findings.

In [2]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from alibi_detect.cd import TabularDrift


---

## 1. Model Evaluation

> Evaluate your model from B1 on data from year 2022 and report the test R2.

In [3]:
df = pd.read_csv('hdb_price_prediction.csv')

# Define the columns to be used
cont_cols = [
    "dist_to_nearest_stn"       , 
    "dist_to_dhoby"             ,     
    "degree_centrality"         , 
    "eigenvector_centrality"    , 
    "remaining_lease_years"     , 
    "floor_area_sqm"
]

cat_cols = [
    "month"             , 
    "town"              ,  
    "flat_model_type"   , 
    "storey_range"
]

target_cols = [
    "resale_price"
]

train   = df[df["year"] <= 2019].copy()
val     = df[df["year"] == 2020].copy()
test    = df[df["year"] == 2022].copy()

In [4]:
from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

data_config = DataConfig(
    target                          = target_cols,  # target column name
    continuous_cols                 = cont_cols,   # list of continuous column names to use
    categorical_cols                = cat_cols,    # list of categorical column names to use
) 

trainer_config = TrainerConfig(
    batch_size      = 1024,
    max_epochs      = 50,
    seed            = SEED,
)

# Define the Model Architecture
model_config = CategoryEmbeddingModelConfig(
    task        = "regression",
    layers      = "5-5-5",     # 3 hidden layer with 5 neurons each
    activation  = "ReLU",
    learning_rate=0.0005
)

optimizer_config = OptimizerConfig(
    optimizer="Adam",
)

tabular_model = TabularModel(
    data_config         = data_config       ,
    model_config        = model_config      ,
    optimizer_config    = optimizer_config  ,
    trainer_config      = trainer_config,
)

2026-03-05 14:47:31,352 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [5]:
import torch

# Capture the original function
_orig_torch_load = torch.load

# Define a version that always sets weights_only to False
def _trusted_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)

# Overwrite the global torch.load
torch.load = _trusted_load

In [6]:
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.preprocessing import StandardScaler

# Define your scaler
target_scaler = StandardScaler()

# Train the model using train/validation split
tabular_model.fit(
    train           =train, 
    validation      =val,
    target_transform=target_scaler
)

# Predict on test set
test_predictions = tabular_model.predict(test)

test_predictions.head()

Seed set to 42
2026-03-05 14:47:31,370 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-05 14:47:31,402 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-05 14:47:31,450 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-05 14:47:31,468 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-05 14:47:31,527 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/leyanzhi/Rep

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │    350 │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.5 K │ train │     0 │
│ 2 │ head             │ LinearHead                │      6 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 1.9 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.9 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_conne
ctor.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: 
UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_conne
ctor.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

2026-03-05 14:48:05,006 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-05 14:48:05,006 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


,resale_price_prediction
116427,246828.640625
116428,309327.125000
116429,277816.031250
116430,301666.750000
116431,318649.625000


In [7]:
# Get prediction column (usually "resale_price_prediction")
predicted_cols = [col for col in test_predictions.columns if col.endswith("_prediction")]
predicted_col = predicted_cols[0]  # one target column

y_true = test[target_cols[0]].to_numpy()
y_pred = test_predictions[predicted_col].to_numpy()

# Metrics
test_mse = mean_squared_error(y_true, y_pred)
test_r2 = r2_score(y_true, y_pred)

print(f"Test MSE: {test_mse:.4f}")
print(f"Test R2 : {test_r2:.4f}")

Test MSE: 17252867874.8608
Test R2 : 0.4048


> Evaluate your model from B1 on data from year 2023 and report the test R2.

In [8]:
test2 = df[df["year"] == 2023].copy()

# Predict on test set
test_predictions2 = tabular_model.predict(test2)

test_predictions2.head()

,resale_price_prediction
143129,213357.437500
143130,243573.875000
143131,237068.562500
143132,243733.078125
143133,211977.421875


In [9]:
predicted_cols2 = [col for col in test_predictions2.columns if col.endswith("_prediction")]
predicted_col2 = predicted_cols2[0]  # one target column

y_true2 = test2[target_cols[0]].to_numpy()
y_pred2 = test_predictions2[predicted_col2].to_numpy()

test2_mse = mean_squared_error(y_true2, y_pred2)
test2_r2 = r2_score(y_true2, y_pred2)

print(f"Test2 MSE: {test2_mse:.4f}")
print(f"Test2 R2 : {test2_r2:.4f}")

Test2 MSE: 25995858905.1714
Test2 R2 : 0.1182


> Did model degradation occur for the deep learning model?


Yes, R2 dropped from 0.4048 to 0.1182



---




## 4. Drifted Features

> Model degradation could be caused by [various data distribution shifts](https://huyenchip.com/2022/02/07/data-distribution-shifts-and-monitoring.html#data-shift-types): covariate shift (features), label shift and/or concept drift (altered relationship between features and labels).
There are various conflicting terminologies in the [literature](https://www.sciencedirect.com/science/article/pii/S0950705122002854#tbl1). Let’s stick to this reference for this assignment.

> Using the **Alibi Detect** library, apply the **TabularDrift** function with the training data (year 2019 and before) used as the reference and **detect which features have drifted** in the 2023 test dataset. Before running the statistical tests, ensure you **sample 800 data points** each from the train and test data. Do not use the whole train/test data. (Hint: use this example as a guide https://docs.seldon.io/projects/alibi-detect/en/stable/examples/cd_chi2ks_adult.html)


The `TabularDrift function` in Alibi Detect is a "two-sample" test wrapper. It applies different statistical tests based on the data type:

- Kolmogorov-Smirnov (KS) Test: Used for your continuous features (`floor_area_sqm`, etc.). It measures the maximum distance between the cumulative distributions of the two samples.

- Chi-Squared Test: Used for your categorical features (`town`, `month`, etc.). It checks if the frequencies of categories have changed significantly.

Covariate Shift: If `is_drift` is 1 (True), it means the distribution of that specific feature has changed significantly between 2019 and 2023. This is a classic Covariate Shift.

The Baseline Effect: Because your reference is 2019 and your test is 2023, features related to time (like `remaining lease`) will almost certainly drift because 4 years have passed for every single flat in the dataset.

P-Value Interpretation: A very low p-value (e.g., $p < 0.001$) suggests that the difference between 2019 and 2023 is not due to random sampling luck, but a genuine shift in the Singapore housing market landscape.

In [13]:
from alibi_detect.cd import TabularDrift
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# 1. Feature Preparation
feature_cols = cont_cols + cat_cols

# Filter for the specific years
train_df = df[df["year"] <= 2019][feature_cols].copy()
test_2023_df = df[df["year"] == 2023][feature_cols].copy()

# 2. SAMPLING FIRST (This is the crucial fix)
train_sample = train_df.sample(n=800, random_state=SEED).copy()
test_sample = test_2023_df.sample(n=800, random_state=SEED).copy()

# 3. Integer Encoding and Category Counting on the samples
categories_per_feature = {}

for col in cat_cols:
    le = LabelEncoder()
    # Fit ONLY on the categories that actually exist in these 1600 rows
    combined_data = pd.concat([train_sample[col], test_sample[col]])
    le.fit(combined_data)
    
    train_sample[col] = le.transform(train_sample[col])
    test_sample[col] = le.transform(test_sample[col])
    
    # Store the exact number of categories present in our samples for Alibi Detect
    col_idx = feature_cols.index(col)
    categories_per_feature[col_idx] = len(le.classes_)

# 4. Convert to Numpy Arrays
X_ref = train_sample.values
X_test_2023 = test_sample.values

# 5. Initialize TabularDrift
cd = TabularDrift(X_ref, p_val=0.05, categories_per_feature=categories_per_feature)

# 6. Run Prediction
drift_preds = cd.predict(X_test_2023)

# 7. Identify which features drifted
features_drifted = []
p_vals = drift_preds['data']['p_val']
threshold = drift_preds['data']['threshold']

for i, feature in enumerate(feature_cols):
    p_value = p_vals[i]
    is_drifted = bool(p_value < threshold)
    if is_drifted:
        features_drifted.append(feature)
    print(f"Feature: {feature:25} | Drifted: {str(is_drifted):5} | p-value: {p_value:.4f}")

print(f"\nTotal features with detected drift: {len(features_drifted)}")
print(f"Overall drift detected: {bool(drift_preds['data']['is_drift'])}")

Feature: dist_to_nearest_stn       | Drifted: False | p-value: 0.4163
Feature: dist_to_dhoby             | Drifted: False | p-value: 0.0155
Feature: degree_centrality         | Drifted: False | p-value: 0.6556
Feature: eigenvector_centrality    | Drifted: False | p-value: 0.0284
Feature: remaining_lease_years     | Drifted: True  | p-value: 0.0000
Feature: floor_area_sqm            | Drifted: False | p-value: 0.1078
Feature: month                     | Drifted: True  | p-value: 0.0000
Feature: town                      | Drifted: False | p-value: 0.0257
Feature: flat_model_type           | Drifted: False | p-value: 0.0116
Feature: storey_range              | Drifted: False | p-value: 0.0758

Total features with detected drift: 2
Overall drift detected: True


> 5. Assuming that the flurry of housing measures have made an impact on the relationship between all the features and resale_price (i.e. P(Y|X) changes), which type of data distribution shift possibly led to model degradation?


This is **Concept Shift**



> 6. From your analysis via TabularDrift, which features contribute to this shift?


> 7. Suggest 1 way to address model degradation and implement it, showing improved test R2 for year 2023.


In [ ]:
# YOUR CODE HERE

---

### Appendix A



Here are our results from a linear regression model. We used StandardScaler for continuous variables and OneHotEncoder for categorical variables.

While 2021 data can be predicted well, test R2 dropped rapidly for 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| Year <= 2020 | 2021     | 0.76    |
| Year <= 2020 | **2022**     | 0.41    |
| Year <= 2020 | **2023**     | **0.10**   |



Similarly, a model trained on 2017 data can predict 2018-2021 well (with slight degradation in performance for 2021), but drops drastically in 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2017         | 2018     | 0.90    |
|              | 2019     | 0.89    |
|              | 2020     | 0.87    |
|              | 2021     | 0.72    |
|              | **2022**     | **0.37**    |
|              | **2023**     | **0.09**    |

With the test set fixed at year 2021, training on data from 2017-2020 still works well on the test data, with minimal degradation. Training sets closer to year 2021 generally do better.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2020         | 2021     | 0.81    |
| 2019         | 2021     | 0.75    |
| 2018         | 2021     | 0.73    |
| 2017         | 2021     | 0.72    |